# Toxicity Detection & Social Media Analysis

## Notebook 05 – Jigsaw Gender/Race Keyword Validation & Toxicity Inference

### Objectives

- Build ground-truth gender/race flags from Jigsaw's own identity annotation columns (already present in `jigsaw_processed.parquet` — nothing was dropped in Notebook 03).
- Apply the keyword lists (`gender.txt`, `race.txt`) to the same text.
- Validate the keyword-based method against the annotation-based ground truth (precision/recall), since Bluesky has **no** identity annotations and will rely on keywords alone.
- Run the trained DeBERTa-v3-base model to get toxicity scores.
- Compare mean toxicity across gender-related / race-related / neither, using both grouping methods.
- Save the combined result as `jigsaw_gender_race_validation.parquet`.



In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import precision_recall_fscore_support, classification_report

from tqdm.auto import tqdm
tqdm.pandas()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)


In [2]:
# ------------------------------------------------------------------
# Local project paths.
# ------------------------------------------------------------------
ROOT = Path("..")

DATASET_DIR = ROOT / "Dataset"
PROCESSED_DIR = DATASET_DIR / "processed"
TRAIN_VALID_SPLIT_DIR = PROCESSED_DIR / "train_valid_split"

JIGSAW_PROCESSED_PATH = PROCESSED_DIR / "jigsaw_processed.parquet"
JIGSAW_BALANCED_PATH = PROCESSED_DIR / "jigsaw_balanced.parquet"
TRAIN_SPLIT_PATH = TRAIN_VALID_SPLIT_DIR / "train.parquet"
VALID_SPLIT_PATH = TRAIN_VALID_SPLIT_DIR / "valid.parquet"

# Trained model
MODELS_DIR = ROOT / "models" / "deberta_v3_base"
MODEL_DIR = MODELS_DIR / "best_model"

# Keyword lists
KEYWORDS_DIR = DATASET_DIR
GENDER_KEYWORDS_PATH = KEYWORDS_DIR / "gender.txt"
RACE_KEYWORDS_PATH = KEYWORDS_DIR / "race.txt"

# Final validation output
OUTPUT_PATH = MODELS_DIR / "predictions" / "jigsaw_gender_race_validation.parquet"

MAX_LENGTH = 384
INFERENCE_BATCH_SIZE = 64

print("Jigsaw processed   :", JIGSAW_PROCESSED_PATH, JIGSAW_PROCESSED_PATH.exists())
print("Jigsaw balanced    :", JIGSAW_BALANCED_PATH, JIGSAW_BALANCED_PATH.exists())
print("Train split        :", TRAIN_SPLIT_PATH, TRAIN_SPLIT_PATH.exists())
print("Valid split        :", VALID_SPLIT_PATH, VALID_SPLIT_PATH.exists())
print("Model dir          :", MODEL_DIR, MODEL_DIR.exists())
print("Gender keywords    :", GENDER_KEYWORDS_PATH, GENDER_KEYWORDS_PATH.exists())
print("Race keywords      :", RACE_KEYWORDS_PATH, RACE_KEYWORDS_PATH.exists())


Jigsaw processed   : /kaggle/input/datasets/tzmughal/social-toxic-sentimental-dataset/jigsaw_processed.parquet True
Jigsaw balanced    : /kaggle/input/datasets/tzmughal/social-toxic-sentimental-dataset/jigsaw_balanced.parquet True
Train split        : /kaggle/input/datasets/tzmughal/social-toxic-sentimental-dataset/train.parquet True
Valid split        : /kaggle/input/datasets/tzmughal/social-toxic-sentimental-dataset/valid.parquet True
Model dir          : /kaggle/input/datasets/tzmughal/social-toxic-sentimental-dataset/deberta_v3_base/deberta_v3_base/best_model True
Gender keywords    : /kaggle/input/datasets/tzmughal/social-toxic-sentimental-dataset/gender.txt True
Race keywords      : /kaggle/input/datasets/tzmughal/social-toxic-sentimental-dataset/race.txt True


### Tokenizer file check

`models/deberta_v3_base/best_model/` contains `tokenizer.json` but not `spm.model` / `vocab.txt` / `merges.txt`. That means only the **fast** tokenizer was saved, even though Notebook 04 loaded the original pretrained tokenizer with `use_fast=False`. So this notebook loads the tokenizer with `AutoTokenizer.from_pretrained(MODEL_DIR)` (no `use_fast=False`) — forcing the slow tokenizer here would fail since its files aren't present. The cell below re-verifies this at runtime rather than assuming it.

In [3]:
# Runtime check — confirms the tokenizer-file situation before we load anything
if MODEL_DIR.exists():
    present = {f.name for f in MODEL_DIR.glob("*")}
    fast_files = [f for f in ["tokenizer.json"] if f in present]
    slow_files = [f for f in ["spm.model", "vocab.txt", "merges.txt"] if f in present]

    print("Files in best_model/:", sorted(present))
    print("Fast-tokenizer files present:", fast_files or "NONE")
    print("Slow-tokenizer files present:", slow_files or "NONE")

    USE_FAST_TOKENIZER = bool(fast_files) and not slow_files
    print("\n--> Loading with use_fast =", USE_FAST_TOKENIZER)
else:
    raise FileNotFoundError(f"MODEL_DIR not found: {MODEL_DIR}. Update the path in the config cell above.")


Files in best_model/: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']
Fast-tokenizer files present: ['tokenizer.json']
Slow-tokenizer files present: NONE

--> Loading with use_fast = True


## Load Processed Jigsaw Data

In [4]:
jigsaw = pd.read_parquet(JIGSAW_PROCESSED_PATH)

print("Shape:", jigsaw.shape)
jigsaw.head(3)


Shape: (1780822, 47)


,id,target,comment_text,severe_toxicity,obscene,identity_attack,insult,threat,asian,atheist,bisexual,black,buddhist,christian,female,heterosexual,hindu,homosexual_gay_or_lesbian,intellectual_or_learning_disability,jewish,latino,male,muslim,other_disability,other_gender,other_race_or_ethnicity,other_religion,other_sexual_orientation,physical_disability,psychiatric_or_mental_illness,transgender,white,created_date,publication_id,parent_id,article_id,rating,funny,wow,sad,likes,disagree,sexual_explicit,identity_annotator_count,toxicity_annotator_count,clean_text,is_toxic
0,59848,0.0,"This is so cool. It's like, 'would you want your mother to read this??' Really great idea, well done!",0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-09-29 10:50:41.987077+00,2,NaN,2006,rejected,0,0,0,0,0,0.0,0,4,"this is so cool. it's like, 'would you want your mother to read this??' really great idea, well done!",0
1,59849,0.0,"Thank you!! This would make my life a lot less anxiety-inducing. Keep it up, and don't let anyone get in your way!",0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-09-29 10:50:42.870083+00,2,NaN,2006,rejected,0,0,0,0,0,0.0,0,4,"thank you!! this would make my life a lot less anxiety-inducing. keep it up, and don't let anyone get in your way!",0
2,59852,0.0,This is such an urgent design problem; kudos to you for taking it on. Very impressive!,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-09-29 10:50:45.222647+00,2,NaN,2006,rejected,0,0,0,0,0,0.0,0,4,this is such an urgent design problem; kudos to you for taking it on. very impressive!,0


## Identity Annotation Ground Truth

Jigsaw's annotators scored each comment for how strongly it mentions each identity group (0–1, NaN where no identity mention was annotated at all). We threshold at `>= 0.5`, the same convention used for `is_toxic` in Notebook 03.

Column groupings:

- **Gender**: `female`, `male`, `transgender`, `other_gender`
- **Race / ethnicity**: `black`, `white`, `asian`, `latino`, `other_race_or_ethnicity`

Sexual orientation columns (`homosexual_gay_or_lesbian`, `bisexual`, `heterosexual`, `other_sexual_orientation`) are intentionally excluded. Adjust the lists below if that scope changes.

In [5]:
GENDER_ANNOT_COLS = ["female", "male", "transgender", "other_gender"]
RACE_ANNOT_COLS = ["black", "white", "asian", "latino", "other_race_or_ethnicity"]

missing_cols = [c for c in GENDER_ANNOT_COLS + RACE_ANNOT_COLS if c not in jigsaw.columns]
assert not missing_cols, f"Missing expected identity columns: {missing_cols}"

jigsaw["gender_related_annot"] = (jigsaw[GENDER_ANNOT_COLS].fillna(0) >= 0.5).any(axis=1).astype(int)
jigsaw["race_related_annot"] = (jigsaw[RACE_ANNOT_COLS].fillna(0) >= 0.5).any(axis=1).astype(int)

jigsaw[["gender_related_annot", "race_related_annot"]].mean()


gender_related_annot    0.045122
race_related_annot      0.021721
dtype: float64

In [6]:
print("Gender-related (annotation-based):")
print(jigsaw["gender_related_annot"].value_counts())

print()
print("Race-related (annotation-based):")
print(jigsaw["race_related_annot"].value_counts())

print()
print("Overlap (both gender and race related):")
print(((jigsaw["gender_related_annot"] == 1) & (jigsaw["race_related_annot"] == 1)).sum())


Gender-related (annotation-based):
gender_related_annot
0    1700467
1      80355
Name: count, dtype: int64

Race-related (annotation-based):
race_related_annot
0    1742141
1      38681
Name: count, dtype: int64

Overlap (both gender and race related):
9196


## Keyword-Based Detection

This replicates the *only* signal available for the Bluesky dataset in the next notebook, so it needs to be validated here against the real annotation labels above.

In [7]:
def load_keywords(path):
    with open(path, "r", encoding="utf-8") as f:
        lines = [line.strip().lower() for line in f]
    return [line for line in lines if line]


def build_pattern(keywords):
    escaped = [re.escape(kw) for kw in keywords]
    pattern = r"\b(?:" + "|".join(escaped) + r")\b"
    return re.compile(pattern, flags=re.IGNORECASE)


In [8]:
gender_keywords = load_keywords(GENDER_KEYWORDS_PATH)
race_keywords = load_keywords(RACE_KEYWORDS_PATH)

print(f"Gender keywords ({len(gender_keywords)}):", gender_keywords)
print(f"Race keywords ({len(race_keywords)}):", race_keywords)

gender_pattern = build_pattern(gender_keywords)
race_pattern = build_pattern(race_keywords)


Gender keywords (37): ['gender', 'sex', 'male', 'female', 'man', 'men', 'woman', 'women', 'boy', 'boys', 'girl', 'girls', 'guy', 'guys', 'lady', 'ladies', 'gentleman', 'gentlemen', 'transgender', 'trans', 'cisgender', 'cis', 'nonbinary', 'non-binary', 'genderqueer', 'genderfluid', 'agender', 'intersex', 'he', 'him', 'his', 'she', 'her', 'hers', 'they', 'them', 'theirs']
Race keywords (40): ['race', 'ethnicity', 'ethnic', 'minority', 'majority', 'immigrant', 'migrant', 'refugee', 'black', 'white', 'asian', 'african', 'african american', 'caucasian', 'hispanic', 'latino', 'latina', 'latinx', 'indigenous', 'native american', 'first nations', 'pacific islander', 'middle eastern', 'arab', 'south asian', 'east asian', 'southeast asian', 'european', 'chinese', 'japanese', 'korean', 'indian', 'pakistani', 'bangladeshi', 'nigerian', 'mexican', 'irish', 'british', 'french', 'german']


In [9]:
# Vectorised regex matching — fast even at ~1.8M rows
jigsaw["gender_related_kw"] = jigsaw["clean_text"].str.contains(gender_pattern, regex=True, na=False).astype(int)
jigsaw["race_related_kw"] = jigsaw["clean_text"].str.contains(race_pattern, regex=True, na=False).astype(int)

print("Gender-related (keyword-based):")
print(jigsaw["gender_related_kw"].value_counts())

print()
print("Race-related (keyword-based):")
print(jigsaw["race_related_kw"].value_counts())


Gender-related (keyword-based):
gender_related_kw
0    941772
1    839050
Name: count, dtype: int64

Race-related (keyword-based):
race_related_kw
0    1645257
1     135565
Name: count, dtype: int64


## Validating the Keyword Lists Against Annotation Ground Truth

This is the key check before trusting the same keyword lists on Bluesky, which has no annotations to fall back on. Precision here means "when the keyword list flags a post, how often is it actually gender/race-related per the annotators." Recall means "of the posts annotators marked gender/race-related, how many did the keyword list catch."

In [10]:
def evaluate_keyword_method(y_true, y_pred, label):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    print(f"--- {label} ---")
    print(classification_report(y_true, y_pred, target_names=["not_related", "related"], digits=4, zero_division=0))
    return {"category": label, "precision": precision, "recall": recall, "f1": f1}


results = []
results.append(evaluate_keyword_method(jigsaw["gender_related_annot"], jigsaw["gender_related_kw"], "Gender"))
results.append(evaluate_keyword_method(jigsaw["race_related_annot"], jigsaw["race_related_kw"], "Race"))

keyword_validation_summary = pd.DataFrame(results)
keyword_validation_summary


--- Gender ---
              precision    recall  f1-score   support

 not_related     0.9985    0.5530    0.7118   1700467
     related     0.0941    0.9825    0.1717     80355

    accuracy                         0.5724   1780822
   macro avg     0.5463    0.7678    0.4418   1780822
weighted avg     0.9577    0.5724    0.6874   1780822

--- Race ---
              precision    recall  f1-score   support

 not_related     0.9982    0.9427    0.9696   1742141
     related     0.2631    0.9222    0.4094     38681

    accuracy                         0.9422   1780822
   macro avg     0.6306    0.9324    0.6895   1780822
weighted avg     0.9822    0.9422    0.9575   1780822



,category,precision,recall,f1
0,Gender,0.094096,0.982528,0.171744
1,Race,0.263128,0.922184,0.409433


### Gender Keyword List — Precision Refinement

The validation above showed gender keyword precision at only ~0.09, driven almost entirely by bare pronouns (`he`, `him`, `his`, `she`, `her`, `hers`, `they`, `them`, `theirs`) that match roughly half of all rows regardless of whether the post is actually gender-related. Race keyword precision was already reasonable and is left unchanged.

This section tests a pruned variant without the bare pronouns against the same annotation ground truth, and automatically adopts whichever version clears a minimum improvement bar.

In [11]:
PRONOUNS_TO_DROP = ["he", "him", "his", "she", "her", "hers", "they", "them", "theirs"]

gender_keywords_trimmed = [kw for kw in gender_keywords if kw not in PRONOUNS_TO_DROP]
print(f"Gender keywords: {len(gender_keywords)} original -> {len(gender_keywords_trimmed)} trimmed")
print("Dropped:", PRONOUNS_TO_DROP)

gender_pattern_trimmed = build_pattern(gender_keywords_trimmed)
jigsaw["gender_related_kw_trimmed"] = jigsaw["clean_text"].str.contains(
    gender_pattern_trimmed, regex=True, na=False
).astype(int)

print("\n--- Comparing original vs. trimmed gender keyword list ---")
gender_result_original = evaluate_keyword_method(
    jigsaw["gender_related_annot"], jigsaw["gender_related_kw"], "Gender (original list)"
)
gender_result_trimmed = evaluate_keyword_method(
    jigsaw["gender_related_annot"], jigsaw["gender_related_kw_trimmed"], "Gender (pronouns dropped)"
)
gender_comparison = pd.DataFrame([gender_result_original, gender_result_trimmed])
print(gender_comparison)

# Decision rule: switch to the trimmed list only on a clear improvement -- precision at least doubles
# AND recall stays at or above 0.75. Below that recall floor, the trimmed list would be silently
# dropping too many genuinely gender-related posts, understating the bias signal rather than fixing it.
PRECISION_MULTIPLIER_REQUIRED = 2.0
RECALL_FLOOR = 0.75

use_trimmed = (
    gender_result_trimmed["precision"] >= gender_result_original["precision"] * PRECISION_MULTIPLIER_REQUIRED
    and gender_result_trimmed["recall"] >= RECALL_FLOOR
)

if use_trimmed:
    jigsaw["gender_related_kw_full"] = jigsaw["gender_related_kw"]  # keep original flag for reference
    jigsaw["gender_related_kw"] = jigsaw["gender_related_kw_trimmed"]
    print(
        f"\n--> DECISION: using trimmed gender keyword list "
        f"(precision {gender_result_original['precision']:.4f} -> {gender_result_trimmed['precision']:.4f}, "
        f"recall {gender_result_original['recall']:.4f} -> {gender_result_trimmed['recall']:.4f}). "
        "'gender_related_kw' now reflects the trimmed list for every cell below; the original-list "
        "flag is kept in 'gender_related_kw_full' for reference."
    )
else:
    print(
        f"\n--> DECISION: keeping the original gender keyword list "
        f"(trimmed precision {gender_result_trimmed['precision']:.4f} didn't clear the "
        f"{PRECISION_MULTIPLIER_REQUIRED}x-improvement / {RECALL_FLOOR} recall-floor bar). "
        "'gender_related_kw' is unchanged. Treat the gender-related toxicity numbers below as noisier "
        "than the race-related ones, and caveat that in the final report."
    )


Gender keywords: 37 original -> 28 trimmed
Dropped: ['he', 'him', 'his', 'she', 'her', 'hers', 'they', 'them', 'theirs']

--- Comparing original vs. trimmed gender keyword list ---
--- Gender (original list) ---
              precision    recall  f1-score   support

 not_related     0.9985    0.5530    0.7118   1700467
     related     0.0941    0.9825    0.1717     80355

    accuracy                         0.5724   1780822
   macro avg     0.5463    0.7678    0.4418   1780822
weighted avg     0.9577    0.5724    0.6874   1780822

--- Gender (pronouns dropped) ---
              precision    recall  f1-score   support

 not_related     0.9982    0.9460    0.9714   1700467
     related     0.4579    0.9646    0.6210     80355

    accuracy                         0.9469   1780822
   macro avg     0.7281    0.9553    0.7962   1780822
weighted avg     0.9739    0.9469    0.9556   1780822

                    category  precision    recall        f1
0     Gender (original list)   0.094096 

## Toxicity Inference

Loads the trained model and scores a subset of Jigsaw. See the caveat at the top of this notebook about which rows are used.

In [12]:
if not VALID_SPLIT_PATH.exists():
    raise FileNotFoundError(
        f"{VALID_SPLIT_PATH} not found. This file was confirmed present during the audit — "
        "check the path/config cell above if this fails."
    )

valid_split = pd.read_parquet(VALID_SPLIT_PATH)
print(f"Loaded {VALID_SPLIT_PATH}: {len(valid_split):,} rows")
print("Columns:", valid_split.columns.tolist())

# Confirmed via 001_full_data_audit.ipynb: valid.parquet has only clean_text + is_toxic,
# no id column. Join on clean_text — check for duplicates first since a collision would
# silently multiply rows in the merge below.
dupe_count = valid_split["clean_text"].duplicated().sum()
if dupe_count:
    print(f"NOTE: {dupe_count} duplicate clean_text value(s) within valid.parquet itself.")

jigsaw_dupe_count = jigsaw["clean_text"].duplicated().sum()
print(f"Duplicate clean_text values in jigsaw_processed: {jigsaw_dupe_count:,} (out of {len(jigsaw):,})")

# jigsaw_processed has thousands of non-unique clean_text values (different original comments
# collapsing to identical cleaned text). Left un-deduplicated on this side, the inner join below
# becomes one-to-many -- a single valid_split row can match multiple jigsaw rows sharing the same
# clean_text, inflating inference_df past len(valid_split) and double/triple counting those posts
# in every downstream group mean and the saved parquet. Dedupe jigsaw on clean_text the same way
# valid_split already is, so the join is one-to-one.
jigsaw_dedup = jigsaw.drop_duplicates(subset="clean_text", keep="first")
print(f"jigsaw_processed rows after clean_text dedup: {len(jigsaw_dedup):,} (dropped {len(jigsaw) - len(jigsaw_dedup):,})")

inference_df = jigsaw_dedup.merge(
    valid_split[["clean_text", "is_toxic"]].drop_duplicates(subset="clean_text"),
    on="clean_text",
    how="inner",
    suffixes=("", "_valid"),
).reset_index(drop=True)

print(f"Matched {len(inference_df):,} rows out of {len(valid_split):,} in valid.parquet.")

if len(inference_df) != len(valid_split):
    print(
        "WARNING: match count differs from valid.parquet's row count. "
        "With jigsaw deduplicated on clean_text, this now most likely means some valid.parquet "
        "clean_text values have no exact match in jigsaw_processed at all (e.g. if jigsaw_processed "
        "was regenerated since the split was made) rather than a one-to-many collision. "
        "Investigate before trusting the toxicity numbers below."
    )

# Sanity check: is_toxic from jigsaw_processed should agree with is_toxic from valid.parquet
# for every matched row, since both were derived the same way from the same target column.
mismatch = (inference_df["is_toxic"] != inference_df["is_toxic_valid"]).sum()
print(f"is_toxic mismatches between jigsaw_processed and valid.parquet on matched rows: {mismatch}")
if mismatch:
    print(
        f"NOTE: these {mismatch} mismatch(es) are a symptom of the same clean_text collisions -- "
        "two different original comments sharing one clean_text can legitimately carry different "
        "is_toxic labels. 'first' occurrence was kept; resolving this fully would need the id "
        "column, which valid.parquet does not have."
    )


Loaded /kaggle/input/datasets/tzmughal/social-toxic-sentimental-dataset/valid.parquet: 20,000 rows
Columns: ['clean_text', 'is_toxic']
NOTE: 13 duplicate clean_text value(s) within valid.parquet itself.
Duplicate clean_text values in jigsaw_processed: 6,603 (out of 1,780,822)
jigsaw_processed rows after clean_text dedup: 1,774,219 (dropped 6,603)
Matched 19,987 rows out of 20,000 in valid.parquet.
is_toxic mismatches between jigsaw_processed and valid.parquet on matched rows: 10
NOTE: these 10 mismatch(es) are a symptom of the same clean_text collisions -- two different original comments sharing one clean_text can legitimately carry different is_toxic labels. 'first' occurrence was kept; resolving this fully would need the id column, which valid.parquet does not have.


## Load Trained Model & Tokenizer

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=USE_FAST_TOKENIZER)
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
model.to(device)
model.eval()

print("Tokenizer type:", type(tokenizer).__name__)
print("Model loaded on:", device)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Tokenizer type: DebertaV2Tokenizer
Model loaded on: cuda


In [14]:
def predict_toxicity(texts, batch_size=INFERENCE_BATCH_SIZE, max_length=MAX_LENGTH):
    probs = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Running inference"):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            list(batch),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            logits = model(**encoded).logits

        batch_probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        probs.extend(batch_probs.tolist())

    return np.array(probs)


texts = inference_df["clean_text"].fillna("").tolist()
inference_df["pred_prob"] = predict_toxicity(texts)
inference_df["pred_label"] = (inference_df["pred_prob"] >= 0.5).astype(int)

inference_df[["clean_text", "is_toxic", "pred_prob", "pred_label"]].head(10)


Running inference:   0%|          | 0/313 [00:00<?, ?it/s]

,clean_text,is_toxic,pred_prob,pred_label
0,"angry trolls, misogynists and racists"", oh my. it doesn't take all of my 150 iq to see the slant here. it's the ""diversity diode"" at work yet agai...",1,0.946507,1
1,interesting concept. how does civil comments plan to monetize their operation?,0,0.000529,0
2,metro has lied to us from the beginning about what this money would really fund. what a pile of lies. makes me sick that my money is paying for th...,1,0.995494,1
3,,0,0.000526,0
4,i noticed there was no point 4.,0,0.000528,0
5,i don't know that this article is discarding mrs. patel. i think it is just stating what moxie tried to pin on her (which was very disgraceful). i...,0,0.012933,0
6,they are terrorists pure and simple.,1,0.995807,1
7,thanks for giving it a chance. :),0,0.000537,0
8,"re ""whatever happened to disneyland"" i was curious and looked it up. it started with phil simms after the super bowl in 1987. he got either <cur>7...",0,0.000706,0
9,and once again the liberals want the decision of a judge (an attorney who once knew a governor) to replace the judgment of a group of scientists. ...,0,0.000547,0


### Cross-check against Notebook 04's saved predictions

`models/deberta_v3_base/predictions/validation_predictions.csv` already has predictions for this exact validation set, saved straight from `trainer.predict(valid_dataset)` — no shuffling in eval, so its row order should match `valid.parquet`'s row order. It has no text column to join on, so this check is positional: if the merge above didn't change row order (it doesn't — `merge` with `how="inner"` on a not-otherwise-sorted key preserves left-frame order here since every `valid_split` row matches at most once), row *i* of this file should equal row *i* of `inference_df`. This confirms the re-run inference above reproduces Notebook 04's original results rather than silently diverging (different padding, truncation, or checkpoint).

## Toxicity by Group — Annotation-Based vs Keyword-Based

In [16]:
def group_summary(df, gender_col, race_col, label):
    def bucket(row):
        if row[gender_col] and row[race_col]:
            return "gender_and_race"
        if row[gender_col]:
            return "gender_only"
        if row[race_col]:
            return "race_only"
        return "neither"

    grouped = df.assign(group=df.apply(bucket, axis=1)).groupby("group")["pred_prob"].agg(["mean", "count"])
    print(f"--- {label} ---")
    print(grouped)
    print()
    return grouped


annot_summary = group_summary(inference_df, "gender_related_annot", "race_related_annot", "Annotation-based grouping")
kw_summary = group_summary(inference_df, "gender_related_kw", "race_related_kw", "Keyword-based grouping")


--- Annotation-based grouping ---
                     mean  count
group                           
gender_and_race  0.837448    206
gender_only      0.688791   1075
neither          0.506876  18045
race_only        0.847630    661

--- Keyword-based grouping ---
                     mean  count
group                           
gender_and_race  0.759401    418
gender_only      0.663039   2061
neither          0.499664  16002
race_only        0.624329   1506



## Save Final Validation Dataset

In [17]:
final_cols = [
    "id", "comment_text", "clean_text", "target", "is_toxic",
    "gender_related_annot", "race_related_annot",
    "gender_related_kw", "gender_related_kw_full", "race_related_kw",
    "pred_prob", "pred_label",
] + GENDER_ANNOT_COLS + RACE_ANNOT_COLS

final_cols = [c for c in final_cols if c in inference_df.columns]

result = inference_df[final_cols].copy()

PROCESSED_DIR.mkdir(exist_ok=True, parents=True)
result.to_parquet(OUTPUT_PATH, index=False)

print("Saved!")
print(OUTPUT_PATH)
print("Shape:", result.shape)


Saved!
/kaggle/working/jigsaw_gender_race_validation.parquet
Shape: (19987, 21)
